# 生成済みキャラへTakaの動作を移す

GitHubのコード一式をGoogle Driveへ配置し、このNotebookから読み込んで実行します。
Notebook内にソースコードや動作データを埋め込まず、教材ZIPのアップロードも行いません。
Takaの調整済み5動作を使用します。たかにゃんの画像やモデルは含みません。

生成済みリグへの移植はCPUで実行できます。

## 1. Google Driveからコードを読み込む

事前に、GitHubのコード一式（tools、reference、motions、integrationなど）をDriveのプロジェクトフォルダーへ配置してください。
PROJECT_DIRは、そのフォルダーを指定します。共有ドライブ内のパスも指定できます。


In [ ]:
from google.colab import drive, output
from pathlib import Path
import sys

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/GameCreation/MotionStudio/10_ImageToCharacter' #@param {type:'string'}
ROOT = Path(PROJECT_DIR).expanduser()
if not (ROOT / 'tools/colab_runtime.py').is_file():
    raise FileNotFoundError('GitHubのコード一式があるフォルダーをPROJECT_DIRに指定してください。')
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / 'tools'))
import importlib, colab_runtime
importlib.reload(colab_runtime)
from colab_runtime import (
    check_project, create_job, run_logged,
    prepare_image, prepare_rig, prepare_web, serve_web, save_result,
)
ROOT = check_project(ROOT)
JOB = create_job(ROOT)
print('コードの読み込み先:', ROOT)
print('今回の作業先:', JOB)


## 2. 実行環境を準備する

環境と重みはColabの一時領域へ置き、Drive上のソースは変更しません。

In [ ]:
ENGINE = Path('/content/motionstudio10/transfer-engine')
run_logged([sys.executable, ROOT / 'tools/setup_transfer.py', '--base', ENGINE], JOB / 'setup.log')
PYTHON = ENGINE / 'venv/bin/python'


## 3. 生成済みのリグを選ぶ

Drive上のGLBまたは結果ZIPのパスを指定します。空欄ならファイル選択画面を開きます。


In [ ]:
RIG_PATH = '' #@param {type:'string'}
RIGGED = prepare_rig(RIG_PATH, JOB)


## Takaの調整済み動作を移す

全体振幅は1.0を保ち、足首の初期向きと関節周辺の変形、接地を補正します。失敗を簡易動作や別キャラで代用しません。

In [ ]:
TRANSFER = JOB / 'transfer'
run_logged([PYTHON, ROOT / 'tools/prepare_transfer.py',
            '--source', RIGGED, '--out', TRANSFER], JOB / 'transfer.log')


## 動作を確認する

歩行、走行、ジャンプを正面、横、背面から確認します。検査警告がある場合は見た目も確認してください。

In [ ]:
WEB = prepare_web(ROOT, TRANSFER, JOB)
if 'AVATAR_SERVER' in globals():
    AVATAR_SERVER.shutdown()
    AVATAR_SERVER.server_close()
AVATAR_SERVER = serve_web(WEB)
output.serve_kernel_port_as_iframe(
    AVATAR_SERVER.server_address[1], height=850, cache_in_notebook=False,
)


## 結果をGoogle Driveへ保存する

PROJECT_DIR/outputs/実行ID/へGLB、診断JSON、ログを含む結果ZIPを保存します。

In [ ]:
ARCHIVE = save_result(ROOT, TRANSFER, JOB)


## PCへもダウンロードする（任意）

In [ ]:
from google.colab import files
files.download(ARCHIVE)
